# Lecture 8: Spread Complexity and fidelity for entangled states

    by M.Süzen
    (c) 2026

Prerequisite to this lecture is finishing the `wigner_semicircle.ipynb`, `wigner_cats.ipynb` and `rozensweig_porter_gap_ratio.ipynb` lectures first for the background. 

The lecture is rather long and theoretical. We detailed the inner workings of Lanczos algorithm, Krylov complexity and fidelity for pure states. 

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
from leymosun.gram import gram_schmidt, is_all_normal, is_all_orthogonal
from leymosun.qcore import (
    prepare_entangled,
    fidelity_evolution_pure_ed,
    fidelity_evolution_pure_krylov,
    spread_complexity_evolution # type: ignore
)
from leymosun.rosenzweig import rp, diagonal_normal
from leymosun.krylov import lanczos, validate_lanczos, get_ritz_matrix

## Gram-Schmidt procedure and QR decomposition

The Gram-Schmidt procedure maps given matrix $A$ (vectors horizontally 
stacked, linearly independent), to $Q$ matrix (vectors), whereby columns 
of $Q$ are orthogonal to each other. 

### Vector projection of $v_{i}$ to $e_{i}$ 

Scalar projection on to $e_{i}$, $\theta$ is the angle between $v_{i}$ and $e_{i}$ ,   
property of dot product: $v_{i} \cdot e_{i} = |v_{i}| |e_{i}| cos \theta$,  
scalar projection is $|v_{i}| cos \theta$ then vector projection is $|v_{i}| \cos \theta \cdot \hat{e_i}$,   
where $\hat{e_{i}} = e_{i}/|e_{i}|$, then vector projection reads $(v_i \cdot \hat{e_i}) \hat{e_i}$

### Iterative projection

At each step we project existing orthonormal bases and remove the resulting projection from the original 
vector iteratively, to find a new orthogonal set of vectors.  

$$ v_k' = v_k - \sum_{i=1}^{k-1} (v_k \cdot \hat{e_i}) \hat{e_i}$$

Then $v_k'$ added to orthogonal bases. 

We use `gram_schmidt` implementation from leymosun and then QR decomposition 
from `numpy` to see that both works. We won't discuss internals of QR decomposition. 

In [ ]:
A = diagonal_normal(100) # This is an orthogonal matrix
Q = gram_schmidt(A)
is_all_normal(Q), is_all_orthogonal(Q), is_all_normal(A), is_all_orthogonal(A)

using numpy 

In [ ]:
import numpy as np
Q, _ = np.linalg.qr(A)
is_all_normal(Q), is_all_orthogonal(Q), is_all_normal(A), is_all_orthogonal(A)

**Exercise 1** Try the `leymosun.gram.gram_schmidt` and `np.linalg.qr` with 
non-orthogonal and non-normalized matrix $B$.

## Krylov Subspace and Complexity: Lanczos Algorithm 

The Krylov subspace method and Lanczos algorithm [lanczos] are related. Lanczos is a method to find eigenvalues of large matrices, but it can also be used for Hilbert space dimensionality reduction for operators, so called `operator growth` can be studied in this context [parker], this concept give rise to
`Spread Complexity` and `Krylov Complexity`, i.e, `operator complexity measures` [bala]. A summary of the approach we take in this lecture is from more of quantum dynamics perspective [camargo]. 
We concentrate on how things are implemented in `leymosun`, with a concise explanations of the the techniques. 

### Exact Diagonalization (ED) Time-independent Schrödinger Equation (SE), Closed System 

The Schrödinger Equation (SE), Closed System expressed as $H | \psi_{n} \rangle = E_{n} | \psi_{n} \rangle $, eigenenergies $E_{n}$, eigenstates $| \psi_{n} \rangle$. $H$ is the Hamiltonian matrix. Finding eigenenergies and eigenstates of $H$ directly using full scale matrix means Exact Diagonalization (ED).

#### Unitary dynamics

State at $t=0$,  An initial state can be express with the basis,  
$| \psi(0) \rangle = \sum_{n} c_{n} | \psi_{n} \rangle$, where $c_{n} = \langle \psi_{n} | \Psi(0) \rangle$ Unitary operator: $ U(t) = e^{-iHt/\hbar}$, where $\hbar=1.0$. 

State evolution: $|\psi(t)\rangle = U(t)|\psi(0)\rangle$, We use **spectral decomposition** to 
compute $U(t)$, essentially $ U(t)= V diag(\exp(-i\cdot t \cdot E_{n} )) V^{\dagger}$ and $V$ is the matrix of eigenstates. $diag$ is placing the vector into diagonals a matrix. 

### Maximally Entangled states

The  maximally entangled quantum state involving $N$ particles.  

This is the most common representation, where all $N$ qubits are in a superposition of the state where all spins are "up" ($|0\rangle$) and the state where all spins are "down" ($|1\rangle$).

For $N$ qubits labeled $1$ to $N$, the state is written as:

$$
| \psi_N \rangle = \frac{1}{\sqrt{2}} \left( |0\rangle^{\otimes N} + |1\rangle^{\otimes N} \right)
$$

Or, expanding the tensor product notation:

$$
|\psi_N\rangle = \frac{1}{\sqrt{2}} \left( \underbrace{|00\dots0\rangle}_{N} + \underbrace{|11\dots1\rangle}_{N} \right)
$$

#### Fidelity

The **Overlap Fidelity**, denoted as $F(t)$, measures the "closeness" between the initial state $|\psi(0)\rangle$ and the evolved state $|\psi(t)\rangle$.

Mathematically, it is defined as the absolute square of their inner product:

$$ F(t) = \left| \langle \psi(0) | \psi(t) \rangle \right|^2 $$

Where:
*   $|\psi(0)\rangle$ is the initial state (e.g., a simple product state).
*   $|\psi(t)\rangle = e^{-iHt} |\psi(0)\rangle$ is the state evolved by the Hamiltonian.

**Range:** $0 \le F(t) \le 1$.
*   $F(t) = 1$: The system has not changed (unitary evolution hasn't mixed the state out of its initial support, or we are at $t=0$).
*   $F(t) = 0$: The states are orthogonal (completely distinguishable).

### Lanczos algorithm : Krylov subspace 

There is a subspace of the Hilbert space that is representative of the initial system, but it has less dimensions (dimensionality reduction). This is the core idea of Krylov subspace and can be computed with Lanczos algorithm. We follow Dirac's notation closely. 

Krylov subspace, Krylov orthonormal bases $|K_{n} \rangle$, We store $b_{n}$, $K_{n}$,  and $a_{n}$.
Where Lanczos coefficients are $a_{n}$ and $b_{n}$.

Initial conditions for $n=0, 1$ must be available before we can iterate in computing    
Krylov bases vectors. We start with $H$ (Hamiltonian matrix) and initial state $\psi(0)$

There is a connection to A=QR decomposition.

* n=0   
$b_{0} = 0$.    
$|K_{0}\rangle = |\psi(0) \rangle$, and       
$a_{0} =  \langle K_{0}|H| K_{0} \rangle$.    
* n=1.    
$|A_{1} \rangle = (H-a_{0} I ) |K_{0} \rangle$       
$b_{1} = \langle A_{1} | A_{1} \rangle^{1/2}$      
$|K_{1} \rangle = b_{1}^{-1} |A_{1} \rangle$     
$a_{1} = \langle K_{1}|H| K_{1} \rangle$     
* n > 1    
$|A_{n+1} \rangle = (H-a_{n} I ) |K_{n} \rangle - b_{n} |K_{n-1} \rangle$.    
$b_{n} = \langle A_{n} | A_{n} \rangle^{1/2}$    
$|K_{n} \rangle = b_{n}^{-1} |A_{n} \rangle$  
$a_{n} = \langle K_{n}|H| K_{n} \rangle$.   
* Lanczos matrix  
$$T_{nm} = \langle K_{n} | H | K_{m} \rangle $$
* Correction checks 
Orthogonality check, $\langle K_{n} | A_{n+1} \rangle = 0$.
$T_{nm}$ is tri-diagonal matrix.  

## Spread Complexity

We define spread complexity as follows. First, we find the Krylov Bases, $|K_{i} \rangle $, these vectors can be used at a given unitary evolution with $|\psi(t) \rangle $, we scaled it with the bases size $n$,  
$C(t) =  \frac{1}{n} \sum_{i=1}^{n} \langle K_{i} | \psi(t) \rangle$.

### Computing fidelity for the unitary chaotic dynamics of pure states

Maximally entangled states can be prepared given number of qubits using leymosun. Using this state, we can compute the fidelity over time. We will do this using exact diagonalization and Krylov subspace.

Exact Diagonalization 

In [ ]:
N=5
gg = 0.1
psi0 = prepare_entangled(N)
H = rp(2**N, gamma=gg)
delta = 1e-3
ndelta = 1000
times, fidelity = fidelity_evolution_pure_ed(H, psi0, delta, ndelta)
plt.plot(times, fidelity)
assert np.sum(fidelity) < 1000

In [ ]:
up_to_dim = 2**N # We cover full Hilbert, but usually Krylov basis can be found much faster. 
a_n, b_n, K_s = lanczos(H, psi0, up_to_dim)
times, fidelity = fidelity_evolution_pure_krylov(K_s, H, psi0, delta, ndelta)
plt.plot(times, fidelity)

validate Lanczos

In [ ]:
r_matrix = get_ritz_matrix(H, up_to_dim, K_s)
validate_lanczos(H, r_matrix)

### Computing Spread Complexity for the unitary chaotic dynamics of entangled states

We can also compute spread complexity [parker], operator growth, for entangled states. 

In [ ]:
N=4 
gg = 1.7  
psi0 = prepare_entangled(N)
up_to_dim = 2**N # We cover full Hilbert, but usually Krylov basis can be found much faster. 
delta = 1.0
ndelta = 50
H = rp(2**N, gamma=gg)
a_n, b_n, K_s = lanczos(H, psi0, up_to_dim)
times, kcs = spread_complexity_evolution(K_s, H, psi0, delta, ndelta)
plt.plot(times, kcs) 

validate Lanczos

In [ ]:
r_matrix = get_ritz_matrix(H, up_to_dim, K_s)
validate_lanczos(H, r_matrix)

**Exercise 2** Compute the fidelity and complexity for regular dynamics, use $gamma=6.5$ in RP.
Did you get consistent results with [suzen26].

## References 

[lanczos] Lanczos, C. (1950), Iteration method for the solution of the eigenvalue problem  
          of linear differential and integral operators. Journal of Research of the National  
          Bureau of Standards. 45 (4): 255–282. doi:10.6028/jres.045.026. 

[parker] D.Parker et al. A Universal Operator Growth Hypothesis, 
         Phys. Rev. X 9, 041017 (2019) [doi](https://doi.org/10.1103/PhysRevX.9.041017)

[bala] Balasubramanian et al., Phys. Rev. D 106, 046007, (2022) [doi](https://doi.org/10.1103/PhysRevD.106.046007).

[camargo] Camargo, H.A., Huh, KB., Jahnke, V. et al. Spread and spectral complexity in   
          quantum spin chains: from integrability to chaos. J. High Energ. Phys.   
          2024, 241 (2024). [doi](https://doi.org/10.1007/JHEP08(2024)241)

[suzen26a] Scrambling of Entanglement from Integrability to Chaos: Bootstrapped        
           Time-Integrated Spread Complexity, M. Suezen, arXiv, 
           [arXiv:2604.14224](https://arxiv.org/abs/2604.14224) (2026).
